# PRIMO TRAINING BERT

In [ ]:
import pandas as pd
import numpy as np
import torch
import wandb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# configuration

EXCEL_PATH    = "Database EcoTSA-Coronaropatia.xlsx"
TEXT_COLUMN   = "REFERTO"
LABEL_COLUMN  = "CORONAROPATIA"

MODEL_NAME = "dbmdz/bert-base-italian-cased"

MAX_LEN    = 512
EVAL_SIZE  = 0.10
BATCH_SIZE = 16
RANDOM_SEED = 42

# phase 1: BERT frozen --> only the classification head is trained
EPOCHS_PHASE1 = 5
LR_PHASE1     = 3e-4  # higher lr since BERT weights are frozen

# phase 2: BERT unfrozen --> full fine-tuning
EPOCHS_PHASE2 = 5
LR_PHASE2     = 2e-5  # much smaller lr to avoid destroying pre-trained weights

In [ ]:
# load and split data

# load data
df = pd.read_excel(EXCEL_PATH, header=1)
df = df.dropna(subset=[LABEL_COLUMN, TEXT_COLUMN])

# force the label column to be an int
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

print(f"Cleaned dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df[LABEL_COLUMN].value_counts()}\n")

# split
train_df, eval_df = train_test_split(
    df,
    test_size=EVAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=df[LABEL_COLUMN]
)

print(f"Train: {len(train_df)}  |  Eval: {len(eval_df)}")

In [ ]:
# class weights

weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df[LABEL_COLUMN]),
    y=train_df[LABEL_COLUMN]
)
class_weight_dict = dict(enumerate(weights_array))
print("Class weights:", {k: round(v, 3) for k, v in class_weight_dict.items()})

weights = torch.tensor(
    [class_weight_dict[i] for i in sorted(class_weight_dict)],
    dtype=torch.float
).to(device)

In [ ]:
# tokenizer
# AutoTokenizer loads the exact tokenizer used to pre-train the model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# dataset helper

def make_dataset(df):
    return Dataset.from_pandas(
        df[[TEXT_COLUMN, LABEL_COLUMN]].rename(
            columns={TEXT_COLUMN: "text", LABEL_COLUMN: "label"}
        ).reset_index(drop=True)
    )

def tokenize(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,        # records longer than MAX_LEN are truncated --> problem
        max_length=MAX_LEN
    )

dataset_train = make_dataset(train_df).map(tokenize, batched=True)
dataset_eval  = make_dataset(eval_df).map(tokenize,  batched=True)

# labels passing through Dataset.map() can silently become int32;
# .long() forces int64 so CrossEntropyLoss does not crash

dataset_train = dataset_train.map(lambda x: {"label": int(x["label"])})
dataset_eval  = dataset_eval.map( lambda x: {"label": int(x["label"])})

# VERIFICA DEI REFERTI TROPPO LUNGHI

In [ ]:
# count tokens for every record using the actual BERT tokenizer
df["bert_token_count"] = df["REFERTO"].apply(
    lambda x: len(tokenizer.tokenize(str(x)))
)

# print only the flagged records with their first 80 characters
print(f"{'Excel row':>10}  {'Tokens':>6}  {'Start of record'}")
print("-" * 90)
for idx, row in df.iterrows():
    if row["bert_token_count"] > 512:
        excel_row = idx + 3
        preview = str(row["REFERTO"])[:80].replace("\n", " ")
        print(f"{excel_row:>10}  {row['bert_token_count']:>6}  {preview}...")
        
print(f"\nTotal records over 512 tokens: {(df['bert_token_count'] > 512).sum()}")

In [ ]:
import nltk
nltk.download('punkt_tab')

# word count using simple whitespace splitting (no subword)
df["word_count"] = df["REFERTO"].apply(lambda x: len(str(x).split()))

# subword token count (already computed)
# df["bert_token_count"] already exists from the previous cell

# ratio for each record
df["subword_ratio"] = df["bert_token_count"] / df["word_count"]

# statistics
print("=== Subword inflation statistics ===")
print(f"Average ratio (bert tokens / words):  {df['subword_ratio'].mean():.3f}")
print(f"Median ratio:                          {df['subword_ratio'].median():.3f}")
print(f"Max ratio:                             {df['subword_ratio'].max():.3f}")
print(f"Min ratio:                             {df['subword_ratio'].min():.3f}")

avg_ratio = df["subword_ratio"].mean()
max_ratio = df["subword_ratio"].max()

# estimated max word count to stay under 512 bert tokens
safe_avg  = int(512 / avg_ratio)
safe_max  = int(512 / max_ratio)

print(f"\n=== Recommended max word count ===")
print(f"Using average ratio ({avg_ratio:.3f}):  stay under ~{safe_avg} words")
print(f"Using worst-case ratio ({max_ratio:.3f}): stay under ~{safe_max} words  (safest)")

In [ ]:
# how many records are under each threshold
under_avg  = (df["word_count"] <= safe_avg).sum()
under_max  = (df["word_count"] <= safe_max).sum()
total      = len(df)

print(f"Records under average-based threshold ({safe_avg} words):    {under_avg}/{total}  ({100*under_avg/total:.1f}%)")
print(f"Records under worst-case threshold ({safe_max} words):  {under_max}/{total}  ({100*under_max/total:.1f}%)")

# Ripresa training

In [ ]:
# weighted trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels.long())
        return (loss, outputs) if return_outputs else loss

In [ ]:
from sklearn.metrics import accuracy_score

# tell the Trainer how to calculate accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # the model outputs raw numbers (logits) --> higher one as prediction
    predictions = np.argmax(logits, axis=-1)
    
    # compare the predictions to the true labels
    acc = accuracy_score(labels, predictions)
    
    # hand it back to the Trainer as a dictionary
    return {"accuracy": acc}

In [ ]:
# frozen BERT

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

print(model) # print classification head

In [ ]:
# freeze all BERT body parameters; only the classification head will update

for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

print("\n── Phase 1: training classification head (BERT frozen) ──")

wandb.init(project="tesi-magistrale", name="normal_run_phase1_freeze")

training_args_phase1 = TrainingArguments(
    output_dir=".results/results_phase1",
    num_train_epochs=EPOCHS_PHASE1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR_PHASE1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    weight_decay=0.0,
    warmup_steps=0,
    lr_scheduler_type="constant",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args_phase1,
    train_dataset=dataset_train,
    eval_dataset=dataset_eval,
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

wandb.finish()

In [ ]:
# unfreeze BERT, full fine-tuning

# re-enable all parameters for end-to-end fine-tuning
for param in model.parameters():
    param.requires_grad = True

print("\n── Phase 2: fine-tuning full model (BERT unfrozen) ──")

wandb.init(project="tesi-magistrale", name="normal_run_phase2_unfreeze")

training_args_phase2 = TrainingArguments(
    output_dir=".results/results_phase2",
    num_train_epochs=EPOCHS_PHASE2,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR_PHASE2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    weight_decay=0.0,
    warmup_steps=0,
    lr_scheduler_type="constant",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args_phase2,
    train_dataset=dataset_train,
    eval_dataset=dataset_eval,
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

wandb.finish()


# GRAFICI (che ora sono anche su wandb)

In [ ]:
import matplotlib.pyplot as plt

# open Hugging Face diary
log_history = trainer.state.log_history

# create empty lists + accuracy
train_loss = []
epochs_train = []

eval_loss = []
eval_accuracy = []
epochs_eval = []

# read through the diary line by line and sort the numbers
for entry in log_history:
    if 'loss' in entry: # Training entry
        train_loss.append(entry['loss'])
        epochs_train.append(entry['epoch'])
    elif 'eval_loss' in entry: # Evaluation entry
        eval_loss.append(entry['eval_loss'])
        eval_accuracy.append(entry['eval_accuracy']) # Grab our new accuracy score!
        epochs_eval.append(entry['epoch'])

# draw the graphs next to each other
plt.figure(figsize=(12, 5)) # Get a wider piece of paper (12 inches wide, 5 inches tall)

# --- Graph 1: loss ---
plt.subplot(1, 2, 1) # 1 row, 2 cols, draw in space 1
plt.plot(epochs_train, train_loss, label='Train', color='blue', marker='o')
plt.plot(epochs_eval, eval_loss, label='Validation', color='orange', marker='o')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True) # faint grid to make reading easier

# --- Graph 2: accuracy ---
plt.subplot(1, 2, 2) 
plt.plot(epochs_eval, eval_accuracy, label='Validation Accuracy', color='green', marker='o')
plt.title('Model Validation Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.grid(True)

plt.tight_layout() # keeps the two graphs from overlapping
plt.show()

# Final evaluation

In [ ]:
# final evaluation

wandb.init(project="tesi-magistrale", name="final_evaluation_run")

print("\n── Final evaluation on held-out set ──")

predictions = trainer.predict(dataset_eval)
wandb.finish()

preds  = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

# checking truncated reports

# add the model's predictions to our eval dataframe
eval_df = eval_df.copy() 
eval_df["model_prediction"] = preds # adds col called "model_prediction" with the data

# recalculate token count for eval_df
eval_df["bert_token_count"] = eval_df[TEXT_COLUMN].apply( # .apply è il "for" loop di pandas, guarda ogni riga dell "text_column"
    lambda x: len(tokenizer.tokenize(str(x))) # prende il testo, lo chiama x, lo rende una stringa, lo tokenizza con il tokenizer di bert, conta la lunghezza dei token e mette quel numero nella colonna "bert_token_count"
)

# create a column to see if the model was right or wrong
eval_df["is_correct"] = eval_df[LABEL_COLUMN] == eval_df["model_prediction"] # confronto tra colonne 

# filter to only show reports longer than 512 tokens
truncated_eval_df = eval_df[eval_df['bert_token_count'] > 512]

# print results
print("\n── Truncated Reports Analysis ──")
print(truncated_eval_df[['model_prediction', 'CORONAROPATIA', 'is_correct', 'bert_token_count']])